# K12 · DreamBooth-LoRA baseline on Stable Diffusion 1.5 (step 5)

One LoRA per image (rank 8, lr 1e-4, **400 steps** (reduced from 600 to finish in < 2 h), batch 1, 512 px, fp16, 8-bit Adam, no prior preservation) for the
8 images in `data/manifests/lora8.csv` (3 natural + 5 OOD), **two trainings at a time (one per T4)**. Then **15** samples
per image (30 DDIM steps, CFG 7.5, seeds 10..39), resized back to the input size.

Progress: one bar per image while training (the diffusers step counter), then one bar per image while sampling.
Limitation (documented): no dev-set tuning of steps / lr on the free tier; the diffusers example's defaults.

**How to run (Kaggle):**
1. Right panel → *Session options*: **Accelerator = GPU T4 x2**, **Internet = On**.
2. Top right **Save Version** → *Save & Run All (Commit)* → Save. It runs in the background (you can close the tab);
   watch it under *View Active Events* / the version's **Logs** (progress lines every 2 min).
3. When it finishes, the result is the version's **Output**: `cs787_k12.zip`.
4. **Not finished?** (it stops itself before 2 hours) → *Add Input* → *Your Work* → this notebook's
   latest version → then *Save Version* again. It continues where it stopped.

In [1]:
import time
T_START = time.time()                      # session clock: work stops before SESSION_HOURS
NOTEBOOK = "K12_lora"
GITHUB_REPO   = "https://github.com/rishabh-vr/CS787-Single-Image-Diffusion-Editor.git"
GITHUB_BRANCH = "main"
SID_COMMIT    = "85381e3c6f29fc07144da854d86be07150404716"     # official code, audited commit
SINDDM_COMMIT = "e7107d9fd198928f26e278f99ec428e261726a45"  # SinDDM, audited commit
SESSION_HOURS = 1.9                        # hard stop: everything (incl. zipping) ends before 2 h

STEPS     = 400
N_SAMPLES = 15
SEED0     = 10
DIFFUSERS_VERSION = "0.35.1"
MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"

In [2]:
import os, sys, subprocess, shutil, glob, json
if not os.path.isdir("/kaggle/working"):
    raise SystemExit("This notebook is written for Kaggle (kaggle.com/code).")
WORK, TMP = "/kaggle/working", "/tmp/cs787"
PROJECT = f"{TMP}/project"
shutil.rmtree(PROJECT, ignore_errors=True)
os.makedirs(TMP, exist_ok=True)
r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GITHUB_BRANCH, GITHUB_REPO, PROJECT],
                   capture_output=True, text=True)
if r.returncode:
    raise SystemExit("Could not download the code from GitHub. Turn Internet ON: right panel -> Session options -> "
                     "Internet (needs a phone-verified Kaggle account), then run again.\n" + r.stderr)
sys.path.insert(0, f"{PROJECT}/src")
os.environ["PYTHONPATH"] = f"{PROJECT}/src" + os.pathsep + os.environ.get("PYTHONPATH", "")
from sidx.kaggle import gather_inputs, unpack_own, pack
from sidx.progress import Bars
print("code:", subprocess.run(["git", "-C", PROJECT, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
gather_inputs()          # results of earlier versions attached under /kaggle/input (resume)

code: cf0386c K12: download + verify the natural images (forest, night_sky, pyramids) before training

[resume] nothing attached under /kaggle/input: starting from scratch


[]

In [3]:
import torch
N_GPUS = torch.cuda.device_count()
if N_GPUS == 0:
    raise SystemExit("No GPU. Right panel -> Session options -> Accelerator -> 'GPU T4 x2', then run again.")
cap = torch.cuda.get_device_capability(0)
if f"sm_{cap[0]}{cap[1]}" not in torch.cuda.get_arch_list():
    raise SystemExit(f"{torch.cuda.get_device_name(0)} is not supported by this PyTorch build: choose 'GPU T4 x2' (not P100).")
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", N_GPUS, "GPU(s) -> all of them are used")

index, name, memory.total [MiB]
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB

torch 2.11.0+cu130 | CUDA 13.0 | 2 GPU(s) -> all of them are used


In [4]:
# only the official repo's data downloader is needed here (no install: keeps K12's diffusers stack untouched)
SID_ROOT = f"{TMP}/efficient-SID-code"
if not os.path.exists(SID_ROOT):
    subprocess.run(["git", "clone", "-q", "https://github.com/haojunqiu/efficient-SID-code", SID_ROOT], check=True)
subprocess.run(["git", "-C", SID_ROOT, "checkout", "-q", SID_COMMIT], check=True)

CompletedProcess(args=['git', '-C', '/tmp/cs787/efficient-SID-code', 'checkout', '-q', '85381e3c6f29fc07144da854d86be07150404716'], returncode=0)

In [5]:
# natural images (balloons, forest, ...) are third-party and not in the repo: fetch + check sha256 against the manifests
r = subprocess.run([sys.executable, "scripts/prepare_natural.py", "--verify", "--sets", "uncond/sinddm15", "--sid-root", SID_ROOT],
                   cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-1200:], r.stderr[-1200:])
assert r.returncode == 0, "data check failed (see above)"

uncond/sinddm15: 15 images verified against data/manifests/uncond_sinddm15.csv
 


In [6]:
!pip uninstall -q -y torchao
!pip install -q diffusers=={DIFFUSERS_VERSION} transformers==4.53.2 accelerate==1.9.0 peft==0.17.1 bitsandbytes pandas
!wget -q -O {TMP}/train_dreambooth_lora.py https://raw.githubusercontent.com/huggingface/diffusers/v{DIFFUSERS_VERSION}/examples/dreambooth/train_dreambooth_lora.py
from huggingface_hub import snapshot_download      # download once, before two trainings start at the same time
snapshot_download(MODEL, allow_patterns=["*.json", "*.txt", "tokenizer/*", "text_encoder/model.safetensors",
                                         "unet/diffusion_pytorch_model.safetensors", "vae/diffusion_pytorch_model.safetensors"])
print("SD 1.5 cached")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

text_encoder/model.safetensors:   0%|          | 0.00/492M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

... (7 outputs left out)

vocab.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

SD 1.5 cached


## Train one LoRA per image, two at a time (skips images already trained)

In [7]:
import csv
from sidx.gpupool import Task, run_pool, _safe
from sidx.progress import last_fraction
unpack_own()
rows = list(csv.DictReader(open(f"{PROJECT}/data/manifests/lora8.csv", encoding="utf-8")))
LORA = f"{WORK}/checkpoints/lora"
LOGS = f"{WORK}/logs/{NOTEBOOK}"
abs_path = lambda r: os.path.normpath(os.path.join(f"{PROJECT}/data/manifests", r["path"]))
weights_ready = lambda i: bool(glob.glob(f"{LORA}/{i}/pytorch_lora_weights.*"))

tasks = []
for r in rows:
    i = r["id"]
    if weights_ready(i):
        continue
    inst = f"{TMP}/lora_inst/{i}"; os.makedirs(inst, exist_ok=True)
    shutil.copy(abs_path(r), f"{inst}/img.png")
    tasks.append(Task(f"train_{i}", [sys.executable, f"{TMP}/train_dreambooth_lora.py", f"--pretrained_model_name_or_path={MODEL}",
        f"--instance_data_dir={inst}", f"--instance_prompt={r['prompt']}", f"--output_dir={TMP}/lora_out/{i}", "--resolution=512",
        "--train_batch_size=1", "--gradient_accumulation_steps=1", "--learning_rate=1e-4", "--lr_scheduler=constant",
        "--lr_warmup_steps=0", f"--max_train_steps={STEPS}", "--rank=8", "--mixed_precision=fp16", "--gradient_checkpointing",
        "--use_8bit_adam", "--checkpointing_steps=100000", "--seed=0"]))

def poll_train():
    res = {}
    for r in rows:
        i = r["id"]
        if weights_ready(i) or glob.glob(f"{TMP}/lora_out/{i}/pytorch_lora_weights.*"):
            res[i] = (STEPS, STEPS)
        else:
            f = last_fraction(f"{LOGS}/{_safe('train_' + i)}.log")
            res[i] = (min(f[0], STEPS), STEPS) if f else (0, STEPS)
    return res

print(len(tasks), "LoRAs to train")
if tasks:
    run_pool(tasks, poll_train, n_gpus=N_GPUS, log_dir=LOGS, unit="step", deadline=T_START + SESSION_HOURS * 3600 - 35 * 60)
for r in rows:                                  # keep the weights in the output
    i = r["id"]
    for f in glob.glob(f"{TMP}/lora_out/{i}/pytorch_lora_weights.*"):
        os.makedirs(f"{LORA}/{i}", exist_ok=True); shutil.copy(f, f"{LORA}/{i}/")
print({r["id"]: weights_ready(r["id"]) for r in rows})

8 LoRAs to train
[pool] 8 tasks on 2 slot(s), 2 GPU(s); logs in /kaggle/working/logs/K12_lora
[10:11:26] forest: 0/400 steps (0%) | 0.0/min | ETA ?
[10:11:26] night_sky: 0/400 steps (0%) | 0.0/min | ETA ?
[10:11:26] pyramids: 0/400 steps (0%) | 0.0/min | ETA ?
[10:11:26] fp_layout: 0/400 steps (0%) | 0.0/min | ETA ?
[10:11:26] fp_house2: 0/400 steps (0%) | 0.0/min | ETA ?
[10:11:26] pat_p4: 0/400 steps (0%) | 0.0/min | ETA ?
[10:11:26] pat_cmm: 0/400 steps (0%) | 0.0/min | ETA ?
[10:11:26] ckt_24aa: 0/400 steps (0%) | 0.0/min | ETA ?
[10:11:26] tasks 0/8 finished | running: GPU0: train_forest (0 min), GPU1: train_night_sky (0 min)
[10:13:26] forest: 129/400 steps (32%) | 64.5/min | ETA 4m12s
[10:13:26] night_sky: 129/400 steps (32%) | 64.5/min | ETA 4m12s
[10:13:26] pyramids: 0/400 steps (0%) | 0.0/min | ETA ?
[10:13:26] fp_layout: 0/400 steps (0%) | 0.0/min | ETA ?
[10:13:26] fp_house2: 0/400 steps (0%) | 0.0/min | ETA ?
[10:13:26] pat_p4: 0/400 steps (0%) | 0.0/min | ETA ?
[10:13:26]

## Sample 30 images per LoRA (live bar per image)

In [8]:
import pandas as pd
from diffusers import StableDiffusionPipeline
from PIL import Image
OUT = f"{WORK}/outputs/phase8_lora/lora_r8_s{STEPS}__lora"
os.makedirs(OUT, exist_ok=True)
if not os.path.exists(f"{OUT}/job.json"):
    json.dump({"experiment": "phase8_lora", "job": f"lora_r8_s{STEPS}", "method": "lora_sd15",
               "params": {"rank": 8, "lr": 1e-4, "steps": STEPS, "resolution": 512, "sampler_steps": 30, "cfg": 7.5},
               "images": [{"id": r["id"], "path": abs_path(r)} for r in rows]}, open(f"{OUT}/job.json", "w"), indent=2)
summary = json.load(open(f"{LOGS}/_summary.json")) if os.path.exists(f"{LOGS}/_summary.json") else {}
cost = {r["id"]: {"train_s": summary.get(f"train_{r['id']}", {}).get("seconds")} for r in rows}
pipe = StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=torch.float16, safety_checker=None).to("cuda")
pipe.set_progress_bar_config(disable=True)
bars = Bars(unit="image", every_s=120)
try:
    for r in rows:
        i = r["id"]
        d = f"{OUT}/{i}"; os.makedirs(d, exist_ok=True)
        have = lambda: sum(os.path.exists(f"{d}/seed_{s:04d}.png") for s in range(SEED0, SEED0 + N_SAMPLES))
        bars.set(i, have(), N_SAMPLES)
        todo = [s for s in range(SEED0, SEED0 + N_SAMPLES) if not os.path.exists(f"{d}/seed_{s:04d}.png")]
        if not todo or not weights_ready(i) or time.time() > T_START + SESSION_HOURS * 3600 - 6 * 60:
            continue
        W, H = Image.open(abs_path(r)).size
        pipe.load_lora_weights(f"{LORA}/{i}")
        torch.cuda.reset_peak_memory_stats(); t0 = time.time()
        for s in todo:
            im = pipe(r["prompt"], num_inference_steps=30, guidance_scale=7.5, generator=torch.Generator("cuda").manual_seed(s)).images[0]
            im.resize((W, H), Image.LANCZOS).save(f"{d}/seed_{s:04d}.png")
            bars.set(i, have(), N_SAMPLES); bars.maybe_print()
        cost[i].update(sample_s_per_img=round((time.time() - t0) / len(todo), 2),
                       sample_peak_vram_gb=round(torch.cuda.max_memory_allocated() / 1024**3, 2))
        pipe.unload_lora_weights()
finally:
    bars.close()
    os.makedirs(f"{WORK}/tables", exist_ok=True)
    json.dump(cost, open(f"{WORK}/tables/phase8_lora_cost.json", "w"), indent=2)
    pack("k12", kinds=("outputs", "checkpoints", "tables"))
pd.DataFrame(cost).T

I0000 00:00:1794911258.330450      23 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/n

[10:27:49] forest: 1/15 images (7%) | 10.1/min | ETA 1m23s


No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


[10:29:53] forest: 15/15 images (100%) | 6.9/min | ETA done
[10:29:53] night_sky: 9/15 images (60%) | 11.2/min | ETA 0m32s


... (5 outputs left out)

[10:36:06] forest: 15/15 images (100%) | 1.8/min | ETA done
[10:36:06] night_sky: 15/15 images (100%) | 2.1/min | ETA done
[10:36:06] pyramids: 15/15 images (100%) | 2.6/min | ETA done
[10:36:06] fp_layout: 15/15 images (100%) | 3.5/min | ETA done
[10:36:06] fp_house2: 15/15 images (100%) | 5.0/min | ETA done
[10:36:06] pat_p4: 15/15 images (100%) | 9.2/min | ETA done
[10:36:06] pat_cmm: 3/15 images (20%) | 11.0/min | ETA 1m05s


No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


[10:38:10] forest: 15/15 images (100%) | 1.4/min | ETA done
[10:38:10] night_sky: 15/15 images (100%) | 1.7/min | ETA done
[10:38:10] pyramids: 15/15 images (100%) | 1.9/min | ETA done
[10:38:10] fp_layout: 15/15 images (100%) | 2.3/min | ETA done
[10:38:10] fp_house2: 15/15 images (100%) | 3.0/min | ETA done
[10:38:10] pat_p4: 15/15 images (100%) | 4.1/min | ETA done
[10:38:10] pat_cmm: 15/15 images (100%) | 6.4/min | ETA done
[10:38:10] ckt_24aa: 11/15 images (73%) | 11.2/min | ETA 0m21s
[10:38:31] forest: 15/15 images (100%) | 1.4/min | ETA done
[10:38:31] night_sky: 15/15 images (100%) | 1.6/min | ETA done
[10:38:31] pyramids: 15/15 images (100%) | 1.9/min | ETA done
[10:38:31] fp_layout: 15/15 images (100%) | 2.2/min | ETA done
[10:38:31] fp_house2: 15/15 images (100%) | 2.8/min | ETA done
[10:38:31] pat_p4: 15/15 images (100%) | 3.7/min | ETA done
[10:38:31] pat_cmm: 15/15 images (100%) | 5.6/min | ETA done
[10:38:31] ckt_24aa: 15/15 images (100%) | 11.1/min | ETA done
[pack] 130

,train_s,sample_s_per_img,sample_peak_vram_gb
forest,272.1,5.44,2.62
night_sky,272.1,5.36,2.62
pyramids,228.1,5.36,2.62
fp_layout,228.1,5.38,2.62
fp_house2,226.1,5.36,2.62
pat_p4,228.1,5.40,2.62
pat_cmm,226.1,5.36,2.62
ckt_24aa,228.1,5.37,2.62


In [9]:
os.makedirs(f"{WORK}/logs", exist_ok=True)
!pip freeze > {WORK}/logs/{NOTEBOOK}_pip_freeze.txt
print("finished; output files:", sorted(os.listdir(WORK)))

finished; output files: ['__notebook__.ipynb', 'cs787_k12.zip', 'logs']
